In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import time
import glob
import random
import copy
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
config = {
    'num_classes': 7,
    'batch_size': 32,
    'num_workers': 4,
    'num_epochs': 50,
    'image_size': 224,
    'initial_lr': 1e-4,
    'best_model_path': "/kaggle/working/efficientnet_unified_finetuned.pth",
    'seed': 42,
    
    # Early Stopping Configuration
    'early_stopping_patience': 5,
    'early_stopping_start_epoch': 25,  # <--- NEW: Starts checking only after 25 epochs
    'min_delta': 0.002,
    
    'weight_decay': 1e-4,
    'model_type': 'efficientnet_b2',
    'unfreeze_last_n_blocks': 3,
}

# ---------------------------------------------------------
# 1. Dataset Class & Logic
# ---------------------------------------------------------
class UnifiedEmotionDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform
        unique_labels = sorted(list(set([item[1] for item in self.file_list])))
        self.classes = unique_labels
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(unique_labels)}

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path, label_name = self.file_list[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            image = Image.new('RGB', (224, 224))
        
        if self.transform:
            image = self.transform(image)
            
        label = self.class_to_idx[label_name]
        return image, label

def get_unified_data_loaders(config):
    print("🔄 Preparing Unified Dataset (Emotion-Rec + AffectNet)...")

    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    all_data = []

    # Part A: Emotion Recognition Dataset
    root_dir_1 = "/kaggle/input/emotion-recognition-dataset/dataset"
    folders_to_keep = ['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise']

    if os.path.exists(root_dir_1):
        print(f"   Loading from {root_dir_1}...")
        for class_name in folders_to_keep:
            folder_path = os.path.join(root_dir_1, class_name)
            images = glob.glob(os.path.join(folder_path, "*")) 
            for img_path in images:
                all_data.append((img_path, class_name))
    else:
        print(f"⚠️ Warning: Directory {root_dir_1} not found.")

    # Part B: AffectNet
    def get_affectnet_files(folder_path, label, oversample=False, target_count=6000):
        valid_extensions = (".jpg", ".png", ".jpeg", ".tiff")
        files = []
        if os.path.exists(folder_path):
            print(f"   Loading {label} from {folder_path}...")
            for filename in os.listdir(folder_path):
                if any(filename.lower().endswith(ext) for ext in valid_extensions):
                    files.append((os.path.join(folder_path, filename), label))
        else:
            print(f"⚠️ Warning: Path {folder_path} does not exist.")
            return []
            
        if oversample and len(files) > 0:
            print(f"   -> Oversampling {label}...")
            oversample_factor = max(1, target_count // len(files))
            final_files = []
            for _ in range(oversample_factor):
                final_files.extend(files)
            final_files.extend(files[:target_count % len(files)])
            return final_files
        return files

    disgust_path = "/kaggle/input/affectnet/archive (3)/Train/disgust"
    fear_path = "/kaggle/input/affectnet/archive (3)/Train/fear"

    all_data.extend(get_affectnet_files(fear_path, 'Fear', oversample=False))
    all_data.extend(get_affectnet_files(disgust_path, 'Disgust', oversample=True, target_count=6000))

    if len(all_data) == 0:
        raise ValueError("No images found! Please check your dataset paths.")

    random.shuffle(all_data)

    total_len = len(all_data)
    train_end = int(0.8 * total_len)
    val_end = int(0.9 * total_len) 

    train_files = all_data[:train_end]
    val_files   = all_data[train_end:val_end]
    test_files  = all_data[val_end:]

    train_dataset = UnifiedEmotionDataset(train_files, transform=train_transforms)
    val_dataset   = UnifiedEmotionDataset(val_files, transform=test_transforms)
    test_dataset  = UnifiedEmotionDataset(test_files, transform=test_transforms)

    pin_memory = torch.cuda.is_available()
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], 
                              shuffle=True, num_workers=config['num_workers'], pin_memory=pin_memory)
    val_loader   = DataLoader(val_dataset, batch_size=config['batch_size'], 
                              shuffle=False, num_workers=config['num_workers'], pin_memory=pin_memory)
    test_loader  = DataLoader(test_dataset, batch_size=config['batch_size'], 
                              shuffle=False, num_workers=config['num_workers'], pin_memory=pin_memory)

    print(f"✅ Data Loaded. Total images: {total_len}")
    print(f"   Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
    return train_loader, val_loader, len(train_dataset), len(val_dataset), len(train_dataset.classes)

# ---------------------------------------------------------
# 2. Training Utilities (Modified EarlyStopping)
# ---------------------------------------------------------
class EarlyStopping:
    """
    Early stops the training if validation loss doesn't improve after a given patience.
    Includes a 'start_epoch' parameter to delay activation.
    """
    def __init__(self, patience=7, min_delta=0, start_epoch=0, verbose=False):
        self.patience = patience
        self.min_delta = min_delta
        self.start_epoch = start_epoch
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf

    def __call__(self, val_loss, model, optimizer, epoch, best_model_path):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, epoch, best_model_path)
        
        elif score < self.best_score + self.min_delta:
            # Only increment counter if we are past the start_epoch
            if epoch >= self.start_epoch:
                self.counter += 1
                print(f'🚨 EarlyStopping counter: {self.counter} out of {self.patience}')
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                if self.verbose:
                    print(f'⚠️ Validation loss did not improve, but Early Stopping is disabled until epoch {self.start_epoch}')
        
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, epoch, best_model_path)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, optimizer, epoch, best_model_path):
        if self.verbose:
            print(f'✅ Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'best_score': self.best_score,
        }, best_model_path)
        self.val_loss_min = val_loss

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

def create_model(config, device):
    model = models.efficientnet_b2(pretrained=True)
    total_blocks = 8 
    
    print(f"🎯 Using {config['model_type']} as backbone")
    
    for param in model.parameters():
        param.requires_grad = False

    unfreeze_blocks = config['unfreeze_last_n_blocks']
    start_unfreeze = max(0, total_blocks - unfreeze_blocks)

    print(f"🔓 Unfreezing last {unfreeze_blocks} blocks")

    for idx, block in enumerate(model.features):
        if idx >= start_unfreeze:
            for param in block.parameters():
                param.requires_grad = True

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, config['num_classes'])
        )

    for param in model.classifier.parameters():
        param.requires_grad = True

    model = model.to(device)
    return model

def train_model(model, train_loader, val_loader, train_size, val_size, device, config):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['initial_lr'],
        weight_decay=config['weight_decay']
    )

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    # Initialize Early Stopping with start_epoch
    early_stopping = EarlyStopping(
        patience=config['early_stopping_patience'],
        min_delta=config['min_delta'],
        start_epoch=config['early_stopping_start_epoch'], # Pass the param here
        verbose=True
    )

    since = time.time()

    for epoch in range(config['num_epochs']):
        print('\n' + '='*60)
        print(f'EPOCH {epoch+1}/{config["num_epochs"]}')
        print('='*60)

        # --- Training ---
        model.train()
        running_loss = 0.0
        running_corrects = 0
        train_pbar = tqdm(train_loader, desc=f'Training Epoch {epoch+1}', unit='batch', ncols=100)

        for batch_idx, (inputs, labels) in enumerate(train_pbar):
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            with torch.set_grad_enabled(True):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data).item()
            
            train_pbar.set_postfix({
                'Loss': f'{running_loss/((batch_idx+1)*config["batch_size"]):.4f}',
                'Acc': f'{running_corrects/((batch_idx+1)*config["batch_size"]):.4f}'
            })

        train_pbar.close()
        epoch_loss = running_loss / train_size
        epoch_acc = running_corrects / train_size
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)

        # --- Validation ---
        model.eval()
        running_loss = 0.0
        running_corrects = 0
        val_pbar = tqdm(val_loader, desc=f'Validation Epoch {epoch+1}', unit='batch', ncols=100)

        for batch_idx, (inputs, labels) in enumerate(val_pbar):
            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data).item()
            
            val_pbar.set_postfix({
                'Loss': f'{running_loss/((batch_idx+1)*config["batch_size"]):.4f}',
                'Acc': f'{running_corrects/((batch_idx+1)*config["batch_size"]):.4f}'
            })

        val_pbar.close()
        val_loss = running_loss / val_size
        val_acc = running_corrects / val_size

        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'📊 Summary - Train: {epoch_loss:.4f}/{epoch_acc:.4f} | Val: {val_loss:.4f}/{val_acc:.4f}')

        if val_acc > best_acc:
            best_acc = val_acc
            print(f'💾 New best accuracy: {best_acc:.4f}')

        # Early stopping check
        early_stopping(val_loss, model, optimizer, epoch, config['best_model_path'])

        if early_stopping.early_stop:
            print("🛑 Early stopping triggered!")
            break

    time_elapsed = time.time() - since
    print(f'\n🎯 Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'🏆 Best validation Accuracy: {best_acc:.4f}')

    checkpoint = torch.load(config['best_model_path'])
    model.load_state_dict(checkpoint['model_state_dict'])

    return history, best_acc, epoch + 1

def plot_results(history):
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], 'b-', label='Training Loss')
    plt.plot(history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], 'b-', label='Training Acc')
    plt.plot(history['val_acc'], 'r-', label='Validation Acc')
    plt.title('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def main():
    set_seed(config['seed'])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("🚀 Using device:", device)
    
    train_loader, val_loader, train_size, val_size, num_classes = get_unified_data_loaders(config)
    config['num_classes'] = num_classes
    
    model = create_model(config, device)
    
    print(f"\n🔥 Starting fine-tuning (Early Stopping active after epoch {config['early_stopping_start_epoch']})...")
    history, best_acc, epochs_trained = train_model(
        model, train_loader, val_loader, train_size, val_size, device, config
    )
    
    plot_results(history)

if __name__ == "__main__":
    main()

🚀 Using device: cuda
🔄 Preparing Unified Dataset (Emotion-Rec + AffectNet)...
   Loading from /kaggle/input/emotion-recognition-dataset/dataset...
   Loading Fear from /kaggle/input/affectnet/archive (3)/Train/fear...
   Loading Disgust from /kaggle/input/affectnet/archive (3)/Train/disgust...
   -> Oversampling Disgust...
✅ Data Loaded. Total images: 21760
   Train: 17408 | Val: 2176 | Test: 2176


🎯 Using efficientnet_b2 as backbone
🔓 Unfreezing last 3 blocks

🔥 Starting fine-tuning (Early Stopping active after epoch 25)...

EPOCH 1/50


📊 Summary - Train: 1.0160/0.6112 | Val: 0.5889/0.7656
💾 New best accuracy: 0.7656
✅ Validation loss decreased (inf --> 0.588884). Saving model...

EPOCH 2/50


📊 Summary - Train: 0.5962/0.7706 | Val: 0.4649/0.8222
💾 New best accuracy: 0.8222
✅ Validation loss decreased (0.588884 --> 0.464920). Saving model...

EPOCH 3/50


📊 Summary - Train: 0.4982/0.8092 | Val: 0.4083/0.8433
💾 New best accuracy: 0.8433
✅ Validation loss decreased (0.464920 --> 0.408289). Saving model...

EPOCH 4/50


📊 Summary - Train: 0.4401/0.8308 | Val: 0.4249/0.8373
⚠️ Validation loss did not improve, but Early Stopping is disabled until epoch 25

EPOCH 5/50


... (23 outputs left out)

📊 Summary - Train: 0.1098/0.9608 | Val: 0.6053/0.8594
🚨 EarlyStopping counter: 3 out of 5

EPOCH 29/50


📊 Summary - Train: 0.1074/0.9636 | Val: 0.5104/0.8686
💾 New best accuracy: 0.8686
🚨 EarlyStopping counter: 4 out of 5

EPOCH 30/50


📊 Summary - Train: 0.1043/0.9628 | Val: 0.5603/0.8594
🚨 EarlyStopping counter: 5 out of 5
🛑 Early stopping triggered!

🎯 Training complete in 52m 32s
🏆 Best validation Accuracy: 0.8686


<Figure size 1500x600 with 2 Axes>

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

# ---------------------------------------------------------
# 1. Create EfficientNet DataFrame (From your logs)
# ---------------------------------------------------------
# I have converted your logs into this CSV string for easy loading
eff_csv_data = """Epoch,Train_Accuracy,Val_Accuracy,Train_Loss,Val_Loss
1,0.6112,0.7656,1.016,0.5889
2,0.7706,0.8222,0.5962,0.4649
3,0.8092,0.8433,0.4982,0.4083
4,0.8308,0.8373,0.4401,0.4249
5,0.8489,0.8465,0.4013,0.3986
6,0.8632,0.8378,0.3619,0.4044
7,0.8722,0.8419,0.341,0.4285
8,0.8866,0.8594,0.3076,0.3942
9,0.894,0.8493,0.281,0.4472
10,0.8992,0.8552,0.2667,0.435
11,0.9074,0.8653,0.2507,0.3947
12,0.9139,0.8612,0.2324,0.4301
13,0.9177,0.8585,0.2208,0.4272
14,0.9235,0.858,0.2069,0.4511
15,0.9278,0.8653,0.1938,0.454
16,0.9307,0.8612,0.1871,0.4749
17,0.9364,0.8649,0.1728,0.4585
18,0.9402,0.8617,0.163,0.468
19,0.9417,0.8575,0.1595,0.4824
20,0.9442,0.8589,0.154,0.4869
21,0.9507,0.8635,0.1361,0.5021
22,0.9519,0.8658,0.1388,0.4377
23,0.9527,0.8594,0.1313,0.4801
24,0.9535,0.8644,0.1282,0.5191
25,0.9561,0.8649,0.1174,0.5466
26,0.9576,0.8649,0.1237,0.5395
27,0.9578,0.8631,0.1183,0.5199
28,0.9608,0.8594,0.1098,0.6053
29,0.9636,0.8686,0.1074,0.5104
30,0.9628,0.8594,0.1043,0.5603"""

eff_df = pd.read_csv(io.StringIO(eff_csv_data))

# ---------------------------------------------------------
# 2. Load MambaVision DataFrame
# ---------------------------------------------------------
mamba_path = "/kaggle/input/metric-csv-mambavision/training_history_epoch0-34.csv"

try:
    mamba_df = pd.read_csv(mamba_path)
    print("✅ MambaVision loaded.")
except FileNotFoundError:
    print(f"❌ File not found: {mamba_path}. Creating dummy data for plot demonstration.")
    # Dummy data just to show the graph works if file is missing
    mamba_df = eff_df.copy()
    mamba_df['Val_Accuracy'] = mamba_df['Val_Accuracy'] - 0.02 # Simulate slightly worse performance
    mamba_df['Val_Loss'] = mamba_df['Val_Loss'] + 0.1

# ---------------------------------------------------------
# 3. Column Mapping (Crucial step)
# ---------------------------------------------------------
# Map Mamba CSV columns to standard names
mamba_cols = {
    'train_acc': 'accuracy',      # Adjust 'accuracy' if your csv has 'train_accuracy' or 'acc'
    'val_acc': 'val_accuracy',    # Adjust 'val_accuracy' if needed
    'train_loss': 'loss',         # Adjust 'loss' if needed
    'val_loss': 'val_loss'
}

# Auto-detect column names for Mamba if they differ
for col in mamba_df.columns:
    if 'val' in col.lower() and 'acc' in col.lower(): mamba_cols['val_acc'] = col
    elif 'acc' in col.lower(): mamba_cols['train_acc'] = col
    elif 'val' in col.lower() and 'loss' in col.lower(): mamba_cols['val_loss'] = col
    elif 'loss' in col.lower(): mamba_cols['train_loss'] = col

# ---------------------------------------------------------
# 4. Generate Appealing Comparison Graph
# ---------------------------------------------------------
sns.set_theme(style="whitegrid", context="talk") # "talk" context makes fonts larger/readable
plt.figure(figsize=(24, 10))

# --- PLOT 1: ACCURACY ---
plt.subplot(1, 2, 1)

# EfficientNet (Blue Palette)
sns.lineplot(data=eff_df, x='Epoch', y='Train_Accuracy', label='EfficientNet (Train)', 
             color='#1f77b4', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=eff_df, x='Epoch', y='Val_Accuracy', label='EfficientNet (Val)', 
             color='#1f77b4', linewidth=4, marker='o')

# MambaVision (Orange Palette)
sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['train_acc'], label='MambaVision (Train)', 
             color='#ff7f0e', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['val_acc'], label='MambaVision (Val)', 
             color='#ff7f0e', linewidth=4, marker='s')

plt.title('🏆 Accuracy Comparison', fontsize=20, weight='bold', pad=20)
plt.xlabel('Epochs', fontsize=16)
plt.ylabel('Accuracy', fontsize=16)
plt.legend(loc='lower right', frameon=True, framealpha=0.9, shadow=True)
plt.grid(True, linestyle=':', alpha=0.6)

# --- PLOT 2: LOSS ---
plt.subplot(1, 2, 2)

# EfficientNet
sns.lineplot(data=eff_df, x='Epoch', y='Train_Loss', label='EfficientNet (Train)', 
             color='#1f77b4', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=eff_df, x='Epoch', y='Val_Loss', label='EfficientNet (Val)', 
             color='#1f77b4', linewidth=4, marker='o')

# MambaVision
sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['train_loss'], label='MambaVision (Train)', 
             color='#ff7f0e', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['val_loss'], label='MambaVision (Val)', 
             color='#ff7f0e', linewidth=4, marker='s')

plt.title('📉 Loss Comparison', fontsize=20, weight='bold', pad=20)
plt.xlabel('Epochs', fontsize=16)
plt.ylabel('Loss', fontsize=16)
plt.legend(loc='upper right', frameon=True, framealpha=0.9, shadow=True)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5. Print Statistical Summary
# ---------------------------------------------------------
best_eff = eff_df['Val_Accuracy'].max()
best_mamba = mamba_df[mamba_cols['val_acc']].max()

print("\n" + "="*50)
print("📊 FINAL VERDICT")
print("="*50)
print(f"EfficientNet Best Val Accuracy: {best_eff:.4f}")
print(f"MambaVision Best Val Accuracy:  {best_mamba:.4f}")
print("-" * 50)
if best_eff > best_mamba:
    print(f"🏆 EfficientNet Wins by +{(best_eff - best_mamba)*100:.2f}%")
else:
    print(f"🏆 MambaVision Wins by +{(best_mamba - best_eff)*100:.2f}%")
print("="*50)

✅ MambaVision loaded.


<Figure size 2400x1000 with 2 Axes>


📊 FINAL VERDICT
EfficientNet Best Val Accuracy: 0.8686
MambaVision Best Val Accuracy:  82.3070
--------------------------------------------------
🏆 MambaVision Wins by +8143.84%


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

# ---------------------------------------------------------
# 1. Create EfficientNet DataFrame (From your logs)
# ---------------------------------------------------------
eff_csv_data = """Epoch,Train_Accuracy,Val_Accuracy,Train_Loss,Val_Loss
1,0.6112,0.7656,1.016,0.5889
2,0.7706,0.8222,0.5962,0.4649
3,0.8092,0.8433,0.4982,0.4083
4,0.8308,0.8373,0.4401,0.4249
5,0.8489,0.8465,0.4013,0.3986
6,0.8632,0.8378,0.3619,0.4044
7,0.8722,0.8419,0.341,0.4285
8,0.8866,0.8594,0.3076,0.3942
9,0.894,0.8493,0.281,0.4472
10,0.8992,0.8552,0.2667,0.435
11,0.9074,0.8653,0.2507,0.3947
12,0.9139,0.8612,0.2324,0.4301
13,0.9177,0.8585,0.2208,0.4272
14,0.9235,0.858,0.2069,0.4511
15,0.9278,0.8653,0.1938,0.454
16,0.9307,0.8612,0.1871,0.4749
17,0.9364,0.8649,0.1728,0.4585
18,0.9402,0.8617,0.163,0.468
19,0.9417,0.8575,0.1595,0.4824
20,0.9442,0.8589,0.154,0.4869
21,0.9507,0.8635,0.1361,0.5021
22,0.9519,0.8658,0.1388,0.4377
23,0.9527,0.8594,0.1313,0.4801
24,0.9535,0.8644,0.1282,0.5191
25,0.9561,0.8649,0.1174,0.5466
26,0.9576,0.8649,0.1237,0.5395
27,0.9578,0.8631,0.1183,0.5199
28,0.9608,0.8594,0.1098,0.6053
29,0.9636,0.8686,0.1074,0.5104
30,0.9628,0.8594,0.1043,0.5603"""

eff_df = pd.read_csv(io.StringIO(eff_csv_data))

# === CRITICAL FIX: Convert EfficientNet accuracy to percentage (0-100) ===
eff_df['Train_Accuracy'] = eff_df['Train_Accuracy'] * 100
eff_df['Val_Accuracy'] = eff_df['Val_Accuracy'] * 100
# =========================================================================


# ---------------------------------------------------------
# 2. Load MambaVision & Map Columns
# ---------------------------------------------------------
mamba_path = "/kaggle/input/metric-csv-mambavision/training_history_epoch0-34.csv"
try:
    mamba_df = pd.read_csv(mamba_path)
    print("✅ MambaVision loaded.")
except FileNotFoundError:
    print(f"❌ File not found: {mamba_path}. Using dummy data for demonstration.")
    mamba_df = eff_df.copy()
    mamba_df['Val_Accuracy'] = mamba_df['Val_Accuracy'] - 2.5
    mamba_df['Val_Loss'] = mamba_df['Val_Loss'] + 0.1

mamba_cols = {
    'train_acc': 'accuracy', 'val_acc': 'val_accuracy',
    'train_loss': 'loss', 'val_loss': 'val_loss'
}

for col in mamba_df.columns:
    if 'val' in col.lower() and 'acc' in col.lower(): mamba_cols['val_acc'] = col
    elif 'acc' in col.lower(): mamba_cols['train_acc'] = col
    elif 'val' in col.lower() and 'loss' in col.lower(): mamba_cols['val_loss'] = col
    elif 'loss' in col.lower(): mamba_cols['train_loss'] = col

# ---------------------------------------------------------
# 3. Generate Zoomed Comparison Graph
# ---------------------------------------------------------
sns.set_theme(style="whitegrid", context="talk")
plt.figure(figsize=(24, 10))

# --- PLOT 1: ACCURACY (ZOOMED) ---
plt.subplot(1, 2, 1)

# Calculate dynamic limits for Accuracy
acc_values = pd.concat([
    eff_df['Train_Accuracy'], eff_df['Val_Accuracy'],
    mamba_df[mamba_cols['train_acc']], mamba_df[mamba_cols['val_acc']]
])
y_min, y_max = acc_values.min(), acc_values.max()
padding = (y_max - y_min) * 0.1

# Plot Lines
sns.lineplot(data=eff_df, x='Epoch', y='Train_Accuracy', label='EfficientNet (Train)', 
             color='#1f77b4', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=eff_df, x='Epoch', y='Val_Accuracy', label='EfficientNet (Val)', 
             color='#1f77b4', linewidth=4, marker='o')

sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['train_acc'], label='MambaVision (Train)', 
             color='#ff7f0e', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['val_acc'], label='MambaVision (Val)', 
             color='#ff7f0e', linewidth=4, marker='s')

plt.title('Accuracy Comparison', fontsize=20, weight='bold', pad=20)
plt.xlabel('Epochs', fontsize=16)
plt.ylabel('Accuracy (%)', fontsize=16) # Changed label to indicate percentage
plt.ylim(y_min - padding, y_max + padding)
plt.legend(loc='lower right', frameon=True, framealpha=0.9, shadow=True)
plt.grid(True, linestyle=':', alpha=0.6)

# --- PLOT 2: LOSS (ZOOMED) ---
plt.subplot(1, 2, 2)

# Calculate dynamic limits for Loss
loss_values = pd.concat([
    eff_df['Train_Loss'], eff_df['Val_Loss'],
    mamba_df[mamba_cols['train_loss']], mamba_df[mamba_cols['val_loss']]
])
l_min, l_max = loss_values.min(), loss_values.max()
l_padding = (l_max - l_min) * 0.1

sns.lineplot(data=eff_df, x='Epoch', y='Train_Loss', label='EfficientNet (Train)', 
             color='#1f77b4', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=eff_df, x='Epoch', y='Val_Loss', label='EfficientNet (Val)', 
             color='#1f77b4', linewidth=4, marker='o')

sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['train_loss'], label='MambaVision (Train)', 
             color='#ff7f0e', linestyle='--', linewidth=2, alpha=0.6)
sns.lineplot(data=mamba_df, x=mamba_df.index + 1, y=mamba_cols['val_loss'], label='MambaVision (Val)', 
             color='#ff7f0e', linewidth=4, marker='s')

plt.title('Loss Comparison', fontsize=20, weight='bold', pad=20)
plt.xlabel('Epochs', fontsize=16)
plt.ylabel('Loss', fontsize=16)
plt.ylim(l_min - l_padding, l_max + l_padding)
plt.legend(loc='upper right', frameon=True, framealpha=0.9, shadow=True)
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

✅ MambaVision loaded.


<Figure size 2400x1000 with 2 Axes>

In [ ]:
import pandas as pd
import seaborn as sns

# ---------------------------------------------------------
# 3. Comparison & Plotting Logic
# ---------------------------------------------------------
def compare_models(efficientnet_history, mamba_csv_path):
    print(f"\n📊 Loading MambaVision data from: {mamba_csv_path}")
    
    # 1. Load MambaVision Data
    try:
        mamba_df = pd.read_csv(mamba_csv_path)
        print("✅ MambaVision CSV loaded.")
    except FileNotFoundError:
        print(f"❌ Error: Could not find CSV at {mamba_csv_path}")
        print("   Skipping comparison plot.")
        return

    # 2. Prepare EfficientNet Data
    eff_df = pd.DataFrame(efficientnet_history)
    
    # 3. Map Columns (Ensure Mamba CSV headers match these keys)
    # Adjust these strings if your CSV headers are different (e.g. 'accuracy' vs 'train_acc')
    mamba_map = {
        'train_acc': 'train_acc', 
        'val_acc': 'val_acc',
        'train_loss': 'train_loss', 
        'val_loss': 'val_loss'
    }
    
    # Auto-detect common column name variations for Mamba
    if 'accuracy' in mamba_df.columns: mamba_map['train_acc'] = 'accuracy'
    if 'val_accuracy' in mamba_df.columns: mamba_map['val_acc'] = 'val_accuracy'
    if 'loss' in mamba_df.columns: mamba_map['train_loss'] = 'loss'

    # 4. Plotting
    plt.figure(figsize=(20, 8))
    sns.set_style("whitegrid")

    # --- Plot A: ACCURACY ---
    plt.subplot(1, 2, 1)
    # EfficientNet
    plt.plot(eff_df['train_acc'], label='EfficientNet (Train)', color='blue', linestyle='--', alpha=0.5)
    plt.plot(eff_df['val_acc'], label='EfficientNet (Val)', color='blue', linewidth=3)
    # MambaVision
    if mamba_map['train_acc'] in mamba_df.columns:
        plt.plot(mamba_df[mamba_map['train_acc']], label='MambaVision (Train)', color='orange', linestyle='--', alpha=0.5)
        plt.plot(mamba_df[mamba_map['val_acc']], label='MambaVision (Val)', color='orange', linewidth=3)
    
    plt.title('Model Accuracy Comparison', fontsize=15, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(fontsize=12)

    # --- Plot B: LOSS ---
    plt.subplot(1, 2, 2)
    # EfficientNet
    plt.plot(eff_df['train_loss'], label='EfficientNet (Train)', color='blue', linestyle='--', alpha=0.5)
    plt.plot(eff_df['val_loss'], label='EfficientNet (Val)', color='blue', linewidth=3)
    # MambaVision
    if mamba_map['train_loss'] in mamba_df.columns:
        plt.plot(mamba_df[mamba_map['train_loss']], label='MambaVision (Train)', color='orange', linestyle='--', alpha=0.5)
        plt.plot(mamba_df[mamba_map['val_loss']], label='MambaVision (Val)', color='orange', linewidth=3)

    plt.title('Model Loss Comparison', fontsize=15, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(fontsize=12)

    plt.tight_layout()
    plt.show()

    # 5. Print Best Scores Table
    print("\n🏆 Final Comparison Summary")
    print("-" * 40)
    print(f"{'Metric':<20} | {'EfficientNet':<15} | {'MambaVision':<15}")
    print("-" * 40)
    
    eff_best_acc = max(eff_df['val_acc'])
    mamba_best_acc = max(mamba_df[mamba_map['val_acc']]) if mamba_map['val_acc'] in mamba_df.columns else 0
    
    print(f"{'Best Val Accuracy':<20} | {eff_best_acc:.4f}          | {mamba_best_acc:.4f}")
    
    winner = "EfficientNet" if eff_best_acc > mamba_best_acc else "MambaVision"
    print("-" * 40)
    print(f"🌟 WINNER: {winner}")

# ---------------------------------------------------------
# 4. Modified Main Execution
# ---------------------------------------------------------
def main():
    set_seed(config['seed'])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("🚀 Using device:", device)
    
    # 1. Load Data
    train_loader, val_loader, train_size, val_size, num_classes = get_unified_data_loaders(config)
    config['num_classes'] = num_classes
    
    # 2. Create Model
    model = create_model(config, device)
    
    # 3. Train (Capture the history variable!)
    print(f"\n🔥 Starting fine-tuning (Early Stopping active after epoch {config['early_stopping_start_epoch']})...")
    history, best_acc, epochs_trained = train_model(
        model, train_loader, val_loader, train_size, val_size, device, config
    )
    
    # 4. Compare with MambaVision
    mamba_csv_path = "/kaggle/input/metric-csv-mambavision/training_history_epoch0-34.csv"
    compare_models(history, mamba_csv_path)

if __name__ == "__main__":
    main()

🚀 Using device: cuda
🔄 Preparing Unified Dataset (Emotion-Rec + AffectNet)...
   Loading from /kaggle/input/emotion-recognition-dataset/dataset...
   Loading Fear from /kaggle/input/affectnet/archive (3)/Train/fear...
   Loading Disgust from /kaggle/input/affectnet/archive (3)/Train/disgust...
   -> Oversampling Disgust...
✅ Data Loaded. Total images: 21760
   Train: 17408 | Val: 2176 | Test: 2176


🎯 Using efficientnet_b2 as backbone
🔓 Unfreezing last 3 blocks

🔥 Starting fine-tuning (Early Stopping active after epoch 25)...

EPOCH 1/50


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score, recall_score
import numpy as np
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(152, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((152, 152)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# ---------------------------------------------------------
# 2. Dataset Class
# ---------------------------------------------------------
class UnifiedEmotionDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform
        
        # Determine classes dynamically from the list
        unique_labels = sorted(list(set([item[1] for item in self.file_list])))
        self.classes = unique_labels
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(unique_labels)}

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path, label_name = self.file_list[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        label = self.class_to_idx[label_name]
        return image, label

# ---------------------------------------------------------
# 3. Build the Master File List
# ---------------------------------------------------------
all_data = []

# --- Part A: Original Dataset (Excluding Ahegao) ---
root_dir_1 = "/kaggle/input/emotion-recognition-dataset/dataset"
folders_to_keep = ['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise']

for class_name in folders_to_keep:
    folder_path = os.path.join(root_dir_1, class_name)
    images = glob.glob(os.path.join(folder_path, "*")) 
    for img_path in images:
        all_data.append((img_path, class_name))

# --- Part B: AffectNet (Adding Disgust & Fear) ---
def get_affectnet_files(folder_path, label, oversample=False, target_count=6000):
    valid_extensions = (".jpg", ".png", ".jpeg", ".tiff")
    files = []
    for filename in os.listdir(folder_path):
        if any(filename.lower().endswith(ext) for ext in valid_extensions):
            files.append((os.path.join(folder_path, filename), label))
            
    if oversample and len(files) > 0:
        oversample_factor = max(1, target_count // len(files))
        final_files = []
        for _ in range(oversample_factor):
            final_files.extend(files)
        # Fill remainder
        final_files.extend(files[:target_count % len(files)])
        return final_files
    return files

# Paths
disgust_path = "/kaggle/input/affectnet/archive (3)/Train/disgust"
fear_path = "/kaggle/input/affectnet/archive (3)/Train/fear"

# Add Fear
all_data.extend(get_affectnet_files(fear_path, 'Fear', oversample=False))
# Add Disgust (Oversampled)
all_data.extend(get_affectnet_files(disgust_path, 'Disgust', oversample=True, target_count=6000))

# ---------------------------------------------------------
# 4. Manual Split & Dataset Creation
# ---------------------------------------------------------

# 1. Shuffle the raw list of strings
random.shuffle(all_data)

# 2. Calculate indices
total_len = len(all_data)
train_end = int(0.8 * total_len)
val_end = int(0.9 * total_len) # 0.8 + 0.1

# 3. Slice the lists
train_files = all_data[:train_end]
val_files   = all_data[train_end:val_end]
test_files  = all_data[val_end:]

# 4. Create Datasets with CORRECT Transforms
# Train gets the heavy augmentation
train_dataset = UnifiedEmotionDataset(train_files, transform=train_transforms)
# Val and Test get the clean resize (No augmentation)
val_dataset   = UnifiedEmotionDataset(val_files, transform=test_transforms)
test_dataset  = UnifiedEmotionDataset(test_files, transform=test_transforms)

# 5. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ---------------------------------------------------------
# 5. Output Verification
# ---------------------------------------------------------
# Note: classes are determined by what is in the list.
# Since we passed lists containing 'Disgust' and 'Fear', they are automatically detected.
print("Classes detected:", train_dataset.classes)
print(f"Total: {len(all_data)}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

NUM_CLASSES = len(train_dataset.classes)
print(f"NUM_CLASSES set to: {NUM_CLASSES}")

In [ ]:
def show_augmented_images(dataloader, emotion_labels, num_images=8):
    def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
        tensor = tensor.clone()
        for t, m, s in zip(tensor, mean, std):
            t.mul_(s).add_(m)
        return tensor

    images, labels = next(iter(dataloader))
    
    images = denormalize(images)

    num_images = min(num_images, images.shape[0])
    
    cols = min(4, num_images)
    rows = math.ceil(num_images / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = axes.ravel() if num_images > 1 else [axes]
    
    for i in range(num_images):
        img = images[i].permute(1, 2, 0).numpy()  # (C,H,W) -> (H,W,C)
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].set_title(emotion_labels[labels[i].item()])
        axes[i].axis('off')
    
    for i in range(num_images, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

show_augmented_images(train_loader, emotion_labels, num_images=8)

In [ ]:
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10):
    train_acc, val_acc, train_loss, val_loss = [], [], [], []
    final_train_preds, final_train_labels = [], []
    final_val_preds, final_val_labels = [], []
    
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        correct, total, running_loss = 0, 0, 0.0
        all_preds, all_labels = [], []
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
        epoch_train_acc = 100 * correct / total
        epoch_train_loss = running_loss / len(train_loader)
        final_train_preds, final_train_labels = all_preds, all_labels

        # --- Validation ---
        model.eval()
        correct, total, running_loss = 0, 0, 0.0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_val_acc = 100 * correct / total
        epoch_val_loss = running_loss / len(val_loader)
        final_val_preds, final_val_labels = all_preds, all_labels
        
        # --- Save history ---
        train_acc.append(epoch_train_acc)
        val_acc.append(epoch_val_acc)
        train_loss.append(epoch_train_loss)
        val_loss.append(epoch_val_loss)
        
        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Acc: {epoch_train_acc:.2f}% | Val Acc: {epoch_val_acc:.2f}% "
              f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    return train_acc, val_acc, train_loss, val_loss, final_train_preds, final_train_labels, final_val_preds, final_val_labels

In [ ]:
EPOCHS = 25
train_acc, val_acc, train_loss, val_loss, train_preds, train_labels, val_preds, val_labels = train_model(model, train_loader, val_loader, EPOCHS)

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    correct, total = 0, 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_acc = 100 * correct / total
    test_error = 100 - test_acc

    print(f"\nTest Accuracy: {test_acc:.2f}% | Test Error: {test_error:.2f}%")
    print("\nTest Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=full_dataset.classes, digits=4))
    
    return all_preds, all_labels, test_acc, test_error

In [ ]:
test_preds, test_labels, test_acc, test_error = evaluate_model(model, test_loader)

In [ ]:
results = {
    "Dataset": ["Training", "Validation", "Testing"],
    "Epochs": [EPOCHS, EPOCHS, EPOCHS],
    "Accuracy (%)": [
        train_acc[-1],
        val_acc[-1],
        test_acc
    ],
    "Error (%)": [
        100 - train_acc[-1],
        100 - val_acc[-1],
        test_error
    ],
    "F1 Score": [
        f1_score(train_labels, train_preds, average="macro"),
        f1_score(val_labels, val_preds, average="macro"),
        f1_score(test_labels, test_preds, average="macro")
    ],
    "Recall": [
        recall_score(train_labels, train_preds, average="macro"),
        recall_score(val_labels, val_preds, average="macro"),
        recall_score(test_labels, test_preds, average="macro")
    ]
}

df_results = pd.DataFrame(results)
print("\n=== Final Performance Table ===")
print(df_results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.preprocessing import label_binarize
import seaborn as sns
import numpy as np
import torch
import os

# # ✅ Create directory for saving plots
# os.makedirs("/kaggle/working/plots", exist_ok=True)

# ✅ Automatically use however many epochs you actually trained
epochs_run = len(val_acc)

# ✅ Use a clean, professional Matplotlib style
plt.style.use('seaborn-v0_8-white')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'legend.fontsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 5
})


# -------------------------------
# 2️⃣ Train vs Validation Accuracy
# -------------------------------
plt.figure(figsize=(7, 5))
plt.plot(range(1, epochs_run + 1), train_acc[:epochs_run], label='Train Acc', color='royalblue')
plt.plot(range(1, epochs_run + 1), val_acc[:epochs_run], label='Val Acc', color='darkorange')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig("train_val_accuracy(EffiNet).png", dpi=300)
plt.show()


# -------------------------------
# 3️⃣ Train vs Validation Loss
# -------------------------------
plt.figure(figsize=(7, 5))
plt.plot(range(1, epochs_run + 1), train_loss[:epochs_run], label='Train Loss', color='royalblue')
plt.plot(range(1, epochs_run + 1), val_loss[:epochs_run], label='Val Loss', color='darkorange')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.savefig("train_val_loss(EffiNet).png", dpi=300)
plt.show()


# -------------------------------
# 4️⃣ Confusion Matrix
# -------------------------------
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=full_dataset.classes, yticklabels=full_dataset.classes)
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()


# -------------------------------
# 5️⃣ Multi-Class ROC Curve (Stylized)
# -------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to NumPy arrays
y_score = np.array(all_probs)
y_true = np.array(all_labels)

# Binarize labels for ROC computation
n_classes = len(np.unique(y_true))
y_true_binarized = label_binarize(y_true, classes=list(range(n_classes)))

# Compute ROC curve and ROC area for each class
fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_binarized.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Plot all ROC curves
plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown'])

for i, color in zip(range(n_classes), colors):
    plt.plot(
        fpr[i],
        tpr[i],
        color=color,
        lw=2,
        label=f"Class {full_dataset.classes[i]} (AUC = {roc_auc[i]:.2f})"
    )

# Micro-average curve
plt.plot(
    fpr["micro"],
    tpr["micro"],
    color='deeppink',
    linestyle=':',
    linewidth=4,
    label=f"Micro-average (AUC = {roc_auc['micro']:.2f})"
)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', lw=2)

# Style and labels
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('Multi-class Receiver Operating Characteristic (ROC)', fontsize=16)
plt.legend(loc="lower right", frameon=False)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

# Save figure
plt.savefig("roc_curve(EffiNet).png", dpi=300)
plt.show()